In [1]:
import torch
import torch.nn  as nn
import math 

In [2]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len,  d_model)
        position = torch.arange(0,  max_len, dtype=torch.float).unsqueeze(1) 
        div_term = torch.exp(torch.arange(0,  d_model, 2).float() * (-math.log(10000.0)/d_model)) 
        pe[:, 0::2] = torch.sin(position  * div_term)
        pe[:, 1::2] = torch.cos(position  * div_term)
        self.register_buffer('pe',  pe.unsqueeze(0))   # [1, max_len, d_model]
 
    def forward(self, x):
        x = x + self.pe[:,  :x.size(1),  :]
        return x
 
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, nhead=8, 
                 num_encoder_layers=6, num_decoder_layers=6, dim_feedforward=2048):
        super().__init__()
        # 输入嵌入层 
        self.src_embed  = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embed  = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_encoder  = PositionalEncoding(d_model)
        
        # Transformer主体
        self.transformer  = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward
        )
        
        # 输出层 
        self.fc_out  = nn.Linear(d_model, tgt_vocab_size)
 
    def forward(self, src, tgt, src_mask=None, tgt_mask=None, d_model=512):
        # 嵌入与位置编码 
        src = self.pos_encoder(self.src_embed(src)  * math.sqrt(d_model)) 
        tgt = self.pos_encoder(self.tgt_embed(tgt)  * math.sqrt(d_model)) 
        
        # 调整维度：[seq_len, batch_size, d_model]
        src = src.permute(1,  0, 2)
        tgt = tgt.permute(1,  0, 2)
        
        # 生成序列掩码 
        if tgt_mask is None:
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(len(tgt)).to(tgt.device) 
        
        # Transformer计算 
        output = self.transformer( 
            src, tgt, 
            src_key_padding_mask=src_mask,
            tgt_key_padding_mask=tgt_mask,
            memory_key_padding_mask=src_mask
        )
        
        # 输出层 
        return self.fc_out(output.permute(1,  0, 2))

In [21]:
batch_size = 50
src_seq_len = 15  # 源语言序列长度
tgt_seq_len = 51  # 目标语言序列长度 
src_vocab = 5000  # 源语言词表大小 
tgt_vocab = 6000  # 目标语言词表大小 
 
# 生成虚拟数据（实际应使用真实分词数据）
src_data = torch.randint(0,  src_vocab, (batch_size, src_seq_len))
tgt_data = torch.randint(0,  tgt_vocab, (batch_size, tgt_seq_len))

In [20]:
device = torch.device("cuda"  if torch.cuda.is_available()  else "cpu")
 
model = Transformer(src_vocab, tgt_vocab).to(device)
optimizer = torch.optim.Adam(model.parameters(),  lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # 假设0为填充符
 
for epoch in range(10):
    optimizer.zero_grad() 
    output = model(src_data.to(device),  tgt_data[:, :-1].to(device))
    
    # 计算损失（忽略最后一个预测位置）
    loss = criterion(
        output.reshape(-1,  tgt_vocab), 
        tgt_data[:, 1:].contiguous().view(-1).to(device)
    )
    
    loss.backward() 
    optimizer.step() 
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}") 

AssertionError: expecting key_padding_mask shape of (50, 14), but got torch.Size([14, 14])